# OpenSearch database explorer

This notebook is independent of the application code. It connects directly to OpenSearch and helps inspect:

- cluster and index status
- index mappings and stored fields
- document counts and sample documents
- metadata, titles, sources, and chunk sizes
- simple charts describing the indexed data

Vector embeddings are excluded from displayed documents because they are large numeric arrays.

In [1]:
import json
import os
from collections import Counter
from getpass import getpass

import altair as alt
import pandas as pd
from IPython.display import display
from opensearchpy import OpenSearch, helpers

alt.data_transformers.disable_max_rows()

# Change these values here, or set the corresponding environment variables.
OPENSEARCH_URL = os.getenv("OPENSEARCH_URL", "http://localhost:9200")
INDEX_NAME = os.getenv("OPENSEARCH_INDEX", "mickey_mouse_wiki_articles_v1")
USERNAME = os.getenv("OPENSEARCH_USERNAME", "admin")
PASSWORD = os.getenv("OPENSEARCH_PASSWORD") or getpass("OpenSearch password: ")

# Limit the number of documents loaded into pandas for visualization.
# The exact index count is still retrieved separately.
MAX_DOCUMENTS_FOR_ANALYSIS = int(os.getenv("OPENSEARCH_VIS_MAX_DOCS", "10000"))

client = OpenSearch(
hosts=[OPENSEARCH_URL],
http_auth=(USERNAME, PASSWORD),
use_ssl=OPENSEARCH_URL.startswith("https://"),
verify_certs=False,
ssl_show_warn=False,
)

print(f"OpenSearch: {OPENSEARCH_URL}")
print(f"Index: {INDEX_NAME}")
print(f"Analysis limit: {MAX_DOCUMENTS_FOR_ANALYSIS:,} documents")

OpenSearch: http://localhost:9200
Index: mickey_mouse_wiki_articles_v1
Analysis limit: 10,000 documents


## 1. Check the connection

In [2]:
cluster_info = client.info()
print(f"Cluster: {cluster_info.get('cluster_name', 'unknown')}")
print(f"Version: {cluster_info.get('version', {}).get('number', 'unknown')}")
print(f"Node: {cluster_info.get('name', 'unknown')}")

if not client.indices.exists(index=INDEX_NAME):
    raise RuntimeError(f"Index '{INDEX_NAME}' does not exist.")

print(f"Index '{INDEX_NAME}' is available.")

Cluster: docker-cluster
Version: 2.13.0
Node: 2a566af8218f
Index 'mickey_mouse_wiki_articles_v1' is available.


## 2. Index health and storage overview

In [3]:
health = client.cluster.health(index=INDEX_NAME)
stats = client.indices.stats(index=INDEX_NAME)
index_stats = stats["indices"][INDEX_NAME]
primary_stats = index_stats.get("primaries", {})

overview = pd.DataFrame([
    {"metric": "health", "value": health.get("status")},
    {"metric": "documents", "value": primary_stats.get("docs", {}).get("count")},
    {"metric": "deleted documents", "value": primary_stats.get("docs", {}).get("deleted")},
    {"metric": "index size", "value": primary_stats.get("store", {}).get("size_in_bytes")},
    {"metric": "shards", "value": health.get("active_shards")},
])
display(overview)

print("\nIndex statistics in JSON:")
display(pd.json_normalize(index_stats, sep="."))

,metric,value
0,health,yellow
1,documents,317
2,deleted documents,0
3,index size,18717035
4,shards,1



Index statistics in JSON:


,uuid,primaries.docs.count,primaries.docs.deleted,primaries.store.size_in_bytes,primaries.store.reserved_in_bytes,primaries.indexing.index_total,primaries.indexing.index_time_in_millis,primaries.indexing.index_current,primaries.indexing.index_failed,primaries.indexing.delete_total,...,total.translog.remote_store.upload.total_upload_size.started_bytes,total.translog.remote_store.upload.total_upload_size.failed_bytes,total.translog.remote_store.upload.total_upload_size.succeeded_bytes,total.request_cache.memory_size_in_bytes,total.request_cache.evictions,total.request_cache.hit_count,total.request_cache.miss_count,total.recovery.current_as_source,total.recovery.current_as_target,total.recovery.throttle_time_in_millis
0,4eVuw2Z6Q1OAxaazERgz0g,317,0,18717035,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## 3. Inspect the mapping

The mapping shows how OpenSearch stores searchable text, metadata, and vector fields.

In [ ]:
mapping_response = client.indices.get_mapping(index=INDEX_NAME)
index_mapping = mapping_response[INDEX_NAME].get("mappings", {})
print(json.dumps(index_mapping, indent=2))

## 4. Count and preview stored documents

The vector field is excluded from the preview so the output stays readable.

In [5]:
total_documents = client.count(index=INDEX_NAME)["count"]
print(f"Total documents in '{INDEX_NAME}': {total_documents:,}")

preview_response = client.search(
    index=INDEX_NAME,
    body={
        "size": min(10, total_documents),
        "_source": {"excludes": ["embedding", "vector"]},
        "query": {"match_all": {}},
    },
)

preview_rows = []
for hit in preview_response["hits"]["hits"]:
    source = hit.get("_source", {})
    metadata = source.get("metadata", {}) or {}
    preview_rows.append({
        "_id": hit.get("_id"),
        "title": metadata.get("title"),
        "pageid": metadata.get("pageid"),
        "chunk_id": metadata.get("chunk_id"),
        "text_preview": str(source.get("text", ""))[:300],
        "metadata": metadata,
    })

preview_df = pd.DataFrame(preview_rows)
display(preview_df)
if preview_rows:
    print("\nComplete first document (without vector data):")
    display(preview_rows[0])

Total documents in 'mickey_mouse_wiki_articles_v1': 317


,_id,title,pageid,chunk_id,text_preview,metadata
0,1b061ba8-e3e3-4325-826a-f0caa832a412,Mickey Mouse,20859,20859_0,Disney cartoon character and mascot \nFor oth...,{'page_url': 'https://en.wikipedia.org/wiki/Mi...
1,ee6e3cf9-428c-4971-91cf-be37eab81ff3,Mickey Mouse,20859,20859_1,* [1 Creation](#Creation)\n+ [1.1 Design](#Des...,{'page_url': 'https://en.wikipedia.org/wiki/Mi...
2,87630c6f-8300-45f2-a6d5-5b54619eb99c,Mickey Mouse,20859,20859_2,[[edit](/w/index.php?title=Mickey_Mouse&action...,{'page_url': 'https://en.wikipedia.org/wiki/Mi...
3,a1878e57-958e-4153-acfd-065fad289c7d,Mickey Mouse,20859,20859_3,[[edit](/w/index.php?title=Mickey_Mouse&action...,{'page_url': 'https://en.wikipedia.org/wiki/Mi...
4,5d01625f-3580-4c5f-bdb4-2310a82f1bab,Mickey Mouse,20859,20859_4,[[edit](/w/index.php?title=Mickey_Mouse&action...,{'page_url': 'https://en.wikipedia.org/wiki/Mi...
5,e285762a-c8cc-4a60-8a99-15265ed0db85,Mickey Mouse,20859,20859_5,[[edit](/w/index.php?title=Mickey_Mouse&action...,{'page_url': 'https://en.wikipedia.org/wiki/Mi...
6,15972066-a51d-4950-9fda-c71e95f3f43c,Mickey Mouse,20859,20859_6,[[edit](/w/index.php?title=Mickey_Mouse&action...,{'page_url': 'https://en.wikipedia.org/wiki/Mi...
7,c54fea51-a798-4d49-9fcb-5d42fa495e05,Mickey Mouse,20859,20859_7,[[edit](/w/index.php?title=Mickey_Mouse&action...,{'page_url': 'https://en.wikipedia.org/wiki/Mi...
8,91512e75-4479-4a6f-8748-bf172533a9a3,Mickey Mouse,20859,20859_8,[[edit](/w/index.php?title=Mickey_Mouse&action...,{'page_url': 'https://en.wikipedia.org/wiki/Mi...
9,8767b24e-4de6-457f-b072-8744e616d3cc,Mickey Mouse,20859,20859_9,[[edit](/w/index.php?title=Mickey_Mouse&action...,{'page_url': 'https://en.wikipedia.org/wiki/Mi...



Complete first document (without vector data):


{'_id': '1b061ba8-e3e3-4325-826a-f0caa832a412',
 'title': 'Mickey Mouse',
 'pageid': '20859',
 'chunk_id': '20859_0',
 'text_preview': 'Disney cartoon character and mascot  \nFor other uses, see [Mickey Mouse (disambiguation)](/wiki/Mickey_Mouse_(disambiguation) "Mickey Mouse (disambiguation)").  \nFictional character  \n| Mickey Mouse | |\n| --- | --- |\n| *[Mickey Mouse & Friends](/wiki/Mickey_Mouse_universe "Mickey Mouse universe")* c',
 'metadata': {'page_url': 'https://en.wikipedia.org/wiki/Mickey_Mouse',
  'source': 'https://en.wikipedia.org/wiki/Mickey_Mouse',
  'title': 'Mickey Mouse',
  'pageid': '20859',
  'chunk_id': '20859_0',
  'revision_id': 1361136931}}

## 5. Load documents for analysis

This uses the OpenSearch scroll helper and excludes embeddings. Increase `MAX_DOCUMENTS_FOR_ANALYSIS` if the index is larger and more data is needed.

In [6]:
records = []
scan_query = {
    "_source": {"excludes": ["embedding", "vector"]},
    "query": {"match_all": {}},
}

for hit in helpers.scan(
    client,
    index=INDEX_NAME,
    query=scan_query,
    size=500,
    scroll="2m",
    preserve_order=False,
):
    source = hit.get("_source", {})
    metadata = source.get("metadata", {}) or {}
    text = str(source.get("text", source.get("page_content", "")) or "")
    records.append({
        "_id": hit.get("_id"),
        "text": text,
        "text_length": len(text),
        "word_count": len(text.split()),
        "title": metadata.get("title", "Unknown"),
        "pageid": metadata.get("pageid"),
        "source": metadata.get("source", "Unknown"),
        "page_url": metadata.get("page_url", "Unknown"),
        "revision_id": metadata.get("revision_id"),
        "chunk_id": metadata.get("chunk_id"),
        "metadata": metadata,
    })
    if len(records) >= MAX_DOCUMENTS_FOR_ANALYSIS:
        break

documents_df = pd.DataFrame(records)
print(f"Loaded {len(documents_df):,} documents for analysis.")
if len(documents_df) < total_documents:
    print(f"Analysis is capped at {MAX_DOCUMENTS_FOR_ANALYSIS:,}; index count remains {total_documents:,}.")
display(documents_df.head(100))

Loaded 317 documents for analysis.


,_id,text,text_length,word_count,title,pageid,source,page_url,revision_id,chunk_id,metadata
0,1b061ba8-e3e3-4325-826a-f0caa832a412,Disney cartoon character and mascot \nFor oth...,6026,703,Mickey Mouse,20859,https://en.wikipedia.org/wiki/Mickey_Mouse,https://en.wikipedia.org/wiki/Mickey_Mouse,1361136931,20859_0,{'page_url': 'https://en.wikipedia.org/wiki/Mi...
1,ee6e3cf9-428c-4971-91cf-be37eab81ff3,* [1 Creation](#Creation)\n+ [1.1 Design](#Des...,1446,146,Mickey Mouse,20859,https://en.wikipedia.org/wiki/Mickey_Mouse,https://en.wikipedia.org/wiki/Mickey_Mouse,1361136931,20859_1,{'page_url': 'https://en.wikipedia.org/wiki/Mi...
2,87630c6f-8300-45f2-a6d5-5b54619eb99c,[[edit](/w/index.php?title=Mickey_Mouse&action...,4926,513,Mickey Mouse,20859,https://en.wikipedia.org/wiki/Mickey_Mouse,https://en.wikipedia.org/wiki/Mickey_Mouse,1361136931,20859_2,{'page_url': 'https://en.wikipedia.org/wiki/Mi...
3,a1878e57-958e-4153-acfd-065fad289c7d,[[edit](/w/index.php?title=Mickey_Mouse&action...,4441,451,Mickey Mouse,20859,https://en.wikipedia.org/wiki/Mickey_Mouse,https://en.wikipedia.org/wiki/Mickey_Mouse,1361136931,20859_3,{'page_url': 'https://en.wikipedia.org/wiki/Mi...
4,5d01625f-3580-4c5f-bdb4-2310a82f1bab,[[edit](/w/index.php?title=Mickey_Mouse&action...,91,4,Mickey Mouse,20859,https://en.wikipedia.org/wiki/Mickey_Mouse,https://en.wikipedia.org/wiki/Mickey_Mouse,1361136931,20859_4,{'page_url': 'https://en.wikipedia.org/wiki/Mi...
...,...,...,...,...,...,...,...,...,...,...,...
95,2a205634-d140-4eaa-8016-446b4883a5bb,[[edit](/w/index.php?title=Mickey_Mouse_Clubho...,2318,279,Mickey Mouse Clubhouse,2300535,https://en.wikipedia.org/wiki/Mickey_Mouse_Clu...,https://en.wikipedia.org/wiki/Mickey_Mouse_Clu...,1363593716,2300535_5,{'page_url': 'https://en.wikipedia.org/wiki/Mi...
96,b147a831-811c-4328-a035-89b91b6f18c0,[[edit](/w/index.php?title=Mickey_Mouse_Clubho...,3458,497,Mickey Mouse Clubhouse,2300535,https://en.wikipedia.org/wiki/Mickey_Mouse_Clu...,https://en.wikipedia.org/wiki/Mickey_Mouse_Clu...,1363593716,2300535_6,{'page_url': 'https://en.wikipedia.org/wiki/Mi...
97,473b614f-8028-493c-8f99-010d1614ab2d,[[edit](/w/index.php?title=Mickey_Mouse_Clubho...,1717,225,Mickey Mouse Clubhouse,2300535,https://en.wikipedia.org/wiki/Mickey_Mouse_Clu...,https://en.wikipedia.org/wiki/Mickey_Mouse_Clu...,1363593716,2300535_7,{'page_url': 'https://en.wikipedia.org/wiki/Mi...
98,28088b40-60d2-4de3-8d4e-a1316c2817d6,[[edit](/w/index.php?title=Mickey_Mouse_Clubho...,809,130,Mickey Mouse Clubhouse,2300535,https://en.wikipedia.org/wiki/Mickey_Mouse_Clu...,https://en.wikipedia.org/wiki/Mickey_Mouse_Clu...,1363593716,2300535_8,{'page_url': 'https://en.wikipedia.org/wiki/Mi...


## 6. Metadata summary

In [ ]:
if documents_df.empty:
    print("No documents were returned.")
else:
    metadata_keys = Counter(
        key
        for metadata in documents_df["metadata"]
        for key in metadata
    )
    metadata_summary = pd.DataFrame(
        metadata_keys.most_common(),
        columns=["metadata_field", "document_count"],
    )
    print("Metadata fields:")
    display(metadata_summary)

    print("Unique values:")
    unique_summary = pd.DataFrame([
        {"field": field, "unique_values": documents_df[field].nunique(dropna=True)}
        for field in ["title", "pageid", "source", "revision_id"]
        if field in documents_df
    ])
    display(unique_summary)

## 7. Documents and chunks by Wikipedia page

Because each Markdown chunk is stored as one OpenSearch document, repeated titles show how many chunks were created for each page.

In [ ]:
if not documents_df.empty:
    title_counts = (
        documents_df["title"]
        .fillna("Unknown")
        .value_counts()
        .head(20)
        .rename_axis("title")
        .reset_index(name="chunks")
    )
    display(title_counts)

    title_chart = (
        alt.Chart(title_counts)
        .mark_bar()
        .encode(
            x=alt.X("chunks:Q", title="Chunk count"),
            y=alt.Y("title:N", sort="-x", title="Wikipedia page"),
            tooltip=["title", "chunks"],
        )
        .properties(title="Top pages by chunk count", height=450)
    )
    display(title_chart)

## 8. Chunk size distribution

In [ ]:
if not documents_df.empty:
    size_summary = documents_df[["text_length", "word_count"]].describe().round(2)
    display(size_summary)

    length_chart = (
        alt.Chart(documents_df)
        .mark_bar()
        .encode(
            x=alt.X("text_length:Q", bin=alt.Bin(maxbins=30), title="Characters per chunk"),
            y=alt.Y("count():Q", title="Number of chunks"),
            tooltip=[alt.Tooltip("count():Q", title="Chunks")],
        )
        .properties(title="Chunk character-length distribution", width=700)
    )
    display(length_chart)

## 9. Source distribution

In [ ]:
if not documents_df.empty:
    source_counts = (
        documents_df["source"]
        .fillna("Unknown")
        .value_counts()
        .head(15)
        .rename_axis("source")
        .reset_index(name="chunks")
    )
    display(source_counts)

    source_chart = (
        alt.Chart(source_counts)
        .mark_bar()
        .encode(
            x=alt.X("chunks:Q", title="Chunk count"),
            y=alt.Y("source:N", sort="-x", title="Source"),
            tooltip=["source", "chunks"],
        )
        .properties(title="Top sources by chunk count", height=350)
    )
    display(source_chart)

## ALERT ALERT !!!!! ------ DELETE BELOW ------ ALERT ALERT !!!!!

In [ ]:
# Permanently delete the configured OpenSearch index.
if client.indices.exists(index=INDEX_NAME):
    response = client.indices.delete(index=INDEX_NAME)
    print(f"Deleted index '{INDEX_NAME}': {response.get('acknowledged', False)}")
else:
    print(f"Index '{INDEX_NAME}' does not exist.")